## Word2Vec
- 문자를 수치형으로 변환 시켜주는 딥러닝 기반의 임베딩 기술 
- 매개변수 
    - sentences
        - 기본값 :  None
        - 토큰화가 된 문장 데이터 (2차원 데이터)
        - None 기본값 ?? -> 학습을 시킬수 있다. 
    - vector_size
        - 기본값 : 100
        - 임베딩 벡터 차원의 개수 ( feature의 수 )
    - window 
        - 기본값 : 5
        - 예측 시 고려할 주변 단어와의 거리 (문맥의 크기)
    - sg
        - 기본값 : 0
        - 0인 경우 
            - CBOW 방식 ( 주변 단어들을 이용하여 중심 단어를 예측 )
        - 1인 경우
            - Skip_gram 방식 (중심 단어를 이용하여 주변 단어를 예측)
        - 빠른 계산이 필요한 경우라면 0을 사용
        - 일반적으로는 1을 사용
    - min_count
        - 기본값 : 5
        - 최소 등장 빈도 수
        - 적게 등장한 단어들을 제외
    - hs
        - 기본값 : 0
        - 계산의 방식 지정 
        - 0 : Negative Sampling (계산량이 적음)
        - 1 : Hierarchical Softmax (계산량 많음)
    - epochs
        - 기본값 :  100
        - 반복 학습 횟수 지정
    - max_vocab_size
        - 기본값 : None
        - 메모리 제한시 사용할 최대 단어의 개수
- 속성 
    - wv
        - 학습된 단어 벡터 (class 형태로 출력)
        - 예 : model.wv['단어']
    - wv.index_to_key
        - 단어의 리스트(학습이 된 단어의 개수) -> 최소 등장 횟수에 영향
        - 등장 빈도 수에 따라 자동 정렬
    - wv.key_to_index
        - 단어 -> 인덱스로 매칭
        - 특정 단어가 인덱스 몇에 위치하는가
    - copus_total_word
        - 전체 학습이 된 단어의 개수 
    - epochs
        - 학습 epoch 수 
    - vector_size
        - 벡터 차원의 수 
- 메서드 
    - wv.most_similar( word, topn = 10 )
        - 특정 단어와 유사한 단어를 출력 
        - topn은 유사한 단어의 개수 지정
    - wv.similarity(word1, word2)
        - 두 단어 간의 코사인 유사도
    - wv.get_vector( word )
        - 특정 단어의 벡터를 반환
    - train()
        - 추가 데이터로 학습 
    - save()
        -  학습된 모델을 저장
    - Word2Vec.load()
        - 저장되어있는 모델을 로드
    

In [ ]:
# 라이브러리 설치 
# !pip install gensim

In [ ]:
from gensim.models import Word2Vec

In [46]:
from sklearn.svm import SVC
from konlpy.tag import Komoran

In [ ]:
docs = [
    '오늘 날씨가 좋다 여행 가고 싶다', 
    '기온이 너무 올라서 아무것도 하기 싫다', 
    '수업이 너무 지루하고 졸리다', 
    '음식이 너무 맛이 없고 서비스도 별로다',
    '영화가 너무 재미있어서 시간이 가는 줄 몰랐다'
]
target = [1, 0, 0, 0, 1]

In [ ]:
# Word2Vec은 토큰화 된 데이터가 필요
komoran = Komoran()

allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']

tokens = []

for doc in docs:
    words = []
    for word, pos in komoran.pos(doc):
        if pos in allow_pos:
            words.append(word)
    tokens.append(words)

tokens

In [ ]:
# Word2Vec을 이용하여 학습 (Skip-gram 방식)
w2v = Word2Vec(
    sentences = tokens, 
    vector_size=100, 
    window = 5, 
    min_count=1, 
    sg = 1, 
    epochs = 100, 
    seed = 42, 
    workers=2
)

In [ ]:
# Word2Vec에서 wv 속성은 객체(class)로 반환 -> 자주 사용 되는 객체임으로 변수에 저장 
wv = w2v.wv

In [ ]:
# wv에 특정 단어를 입력하면 벡터 출력
wv['여행']

In [ ]:
# 유사한 단어 찾기 
wv.most_similar('음식', topn = 3)

In [ ]:
# 두 단어의 코사인 유사도를 확인 
wv.similarity('여행', '음식')

In [ ]:
len(wv.index_to_key)

In [ ]:
import numpy as np

In [ ]:
# tokens의 단어들 중 w2v의 index_to_key에 존재하는 데이터의 단위 벡터를 확인 
vectors = []

for token in tokens:
    for word in token:
        # print(word)
        vec = []
        if word in wv.index_to_key:
            # print(word)
            # tokens 데이터에서 단어가 w2v의 학습 단어에 포함되어있을때
            # 해당 단어의 벡터 값을 vec에 추가 
            vec.append(wv[word])
            # print(wv[word].shape)
    
    vectors.append( np.mean(vec, axis=0) )
    #     break
    # break    
vectors

In [57]:
np.array(vectors).shape

(5, 100)

In [47]:
svc = SVC(random_state=42)

In [50]:
target

[1, 0, 0, 0, 1]

In [54]:
svc.fit(np.array(vectors), target)

UnicodeDecodeError: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence

UnicodeDecodeError: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence

SVC(random_state=42)